# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the [mlcroissant](https://github.com/mlcommons/croissant) library.

This dataset provides key demographic and psychological indicator data collected via surveys among residents of Kilifi County, Kenya. It is described using the [Croissant](https://mlcommons.org/croissant/) standard with all fields, record sets, and columns referenced by their canonical `@id` values for reproducible, standard-compliant access.

### Dataset Source
The dataset source is defined by the Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install `mlcroissant` if needed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata and description
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview

Examine available **Record Sets** and their associated **Fields** using their `@id`. All primary entities in the Croissant schema are referenced by their `@id` property. This allows you to interact with the schema and data programmatically and without ambiguity.

Let's print an overview of all `recordSet` objects (`@id`s and their field IDs).

In [ ]:
# List available record sets, their IDs and associated fields
record_sets = list(dataset.metadata.record_sets)
print(f"Record sets found: {len(record_sets)}\n")
all_record_set_ids = []

for rs in record_sets:
    print(f"- Record set: {rs.name} (@id: {rs.id})")
    all_record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print('')

## 3. Data Extraction

Let's extract data from each record set and load it into a pandas DataFrame for further analysis. We'll use the precise `@id` for each record set and field, as required by the Croissant specification.

> **Note:** If the dataset has multiple record sets, we load all available record sets into a dictionary of DataFrames, with their `@id` as the keys for easy reference.

In [ ]:
dataframes = {}

# Use the record set IDs discovered earlier
for record_set_id in all_record_set_ids:
    # Use Croissant Dataset.records() loader by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show an example: first few columns and rows of the first record set
if all_record_set_ids:
    first_set_id = all_record_set_ids[0]
    print(f"Fields for record set {@id}: {first_set_id}")
    print(dataframes[first_set_id].columns.tolist())
    display(dataframes[first_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Perform data analysis steps such as filtering records, normalizing fields, grouping, etc., using **only the field `@id`s** in DataFrame column access and operations.

In this example, we select a numeric field (e.g., GAD-7 score, PHQ-9 score, or age) from the extracted DataFrame and perform basic EDA: filter by threshold, normalize, and group by a demographic field (e.g., gender or education level).

In [ ]:
# Adjust these to match your dataset schema IDs (adjust @id as needed after viewing above)
# Examples, replace if they differ:
record_set_id = all_record_set_ids[0]  # Use the first record set
df = dataframes[record_set_id]

# Guess likely '@id's for the numeric and categorical fields (adjust to known @id values from above)
# For example:
#   GAD-7 Score: 'gad_7_score', Age: 'age', Gender: 'gender', PHQ-9: 'phq_9_score', Education: 'level_of_education'

# Try auto-detect numeric fields for demonstration purposes
numeric_fields = [col for col in df.columns if (df[col].dtype in [int, float]) or (pd.api.types.is_numeric_dtype(df[col]))]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # fallback if all columns are string/object
    numeric_field_id = df.columns[0]
# For grouping, find first non-numeric field
group_fields = [col for col in df.columns if col != numeric_field_id]
group_field_id = group_fields[0] if group_fields else None

print(f"Numeric field selected for analysis: {numeric_field_id}")
if group_field_id:
    print(f"Grouping field: {group_field_id}")

# Try to filter on values greater than the mean of this numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id]-filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group by group_field_id (if string/object type)
    if group_field_id and group_field_id in filtered_df.columns and not pd.api.types.is_numeric_dtype(filtered_df[group_field_id]):
        grouped_df = filtered_df.groupby(group_field_id).agg({numeric_field_id:'mean', normalized_col:'mean'})
        print(f"Mean {numeric_field_id} and normalized by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field available (all data are string/categorical).")

## 5. Visualization

Visualize data distributions or relationships between fields. Below, we plot the distribution of a numeric field and, if available, visualize differences by group (using `@id` columns only).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns and not pd.api.types.is_numeric_dtype(df[group_field_id]):
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We have:
- Loaded metadata and record sets from the FAIR² Mental Health Survey Dataset using `mlcroissant`
- Inspected data schema including all `@id`s for record sets and fields
- Loaded data into DataFrames using the Croissant-compliant loader
- Filtered and normalized numeric fields and grouped by relevant taxonomy
- Visualized key distributions

**Further analysis** can include correlating mental health scores with specific demographics, handling missing data, advanced visualizations, or exporting prepared records for downstream machine learning tasks.

> All referenced schema elements are accessed via their canonical `@id` as required for FAIR, reproducible research.